# Fraud Shield — Week 4 Bronze Ingestion

**Project:** Fraud Shield: Transaction Risk Monitoring  
**Target file:** `notebooks/02_bronze_ingestion.ipynb`

This is a fresh notebook for controlled batch Source-to-Bronze ingestion of the five approved source files. Source business values are preserved; Bronze metadata is added for lineage and auditing.


## 1. Environment Setup

The existing Unity Catalog environment is used. No new schema is created.


In [ ]:
%sql
USE CATALOG workspace;
USE SCHEMA default;

SELECT current_catalog() AS catalog, current_schema() AS schema;


In [ ]:
volume_path = "/Volumes/Data_Engineering/default/myvolume_01"
print("Volume:", volume_path)
display(dbutils.fs.ls(volume_path))


## 2. Source Inventory

| Source file | Format | Bronze table |
|---|---|---|
| transactions_databricks_compatible.parquet | Parquet | bronze_transactions |
| customers.json | JSON | bronze_customers |
| devices.csv | CSV | bronze_devices |
| merchants.csv | CSV | bronze_merchants |
| fraud_cases.csv | CSV | bronze_fraud_cases |


## 3. Create Temporary Source Views

Databricks SQL is used for Bronze creation and verification. Small PySpark reader cells are used only to expose the existing Volume files as temporary views because the project contains mixed Parquet, JSON and CSV formats.


In [ ]:
# Transactions source
df_transactions = spark.read.parquet(
    f"{volume_path}/transactions_databricks_compatible.parquet"
)
df_transactions.createOrReplaceTempView("src_transactions")

display(df_transactions.limit(10))
df_transactions.printSchema()
print("Transactions source count:", df_transactions.count())


In [ ]:
# Customers source
df_customers = spark.read.json(
    f"{volume_path}/customers.json"
)
df_customers.createOrReplaceTempView("src_customers")

display(df_customers.limit(10))
df_customers.printSchema()
print("Customers source count:", df_customers.count())


In [ ]:
# Devices source
df_devices = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(f"{volume_path}/devices.csv")
)
df_devices.createOrReplaceTempView("src_devices")

display(df_devices.limit(10))
df_devices.printSchema()
print("Devices source count:", df_devices.count())


In [ ]:
# Merchants source
df_merchants = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(f"{volume_path}/merchants.csv")
)
df_merchants.createOrReplaceTempView("src_merchants")

display(df_merchants.limit(10))
df_merchants.printSchema()
print("Merchants source count:", df_merchants.count())


In [ ]:
# Fraud Cases source
df_fraud_cases = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(f"{volume_path}/fraud_cases.csv")
)
df_fraud_cases.createOrReplaceTempView("src_fraud_cases")

display(df_fraud_cases.limit(10))
df_fraud_cases.printSchema()
print("Fraud Cases source count:", df_fraud_cases.count())


# 4. Transactions → Bronze

The SQL statement preserves all source columns and adds Bronze ingestion metadata.


In [ ]:
%sql
CREATE OR REPLACE TABLE workspace.default.bronze_transactions
USING DELTA
AS
SELECT
    t.*,
    'transactions_databricks_compatible.parquet' AS _source_file,
    current_timestamp() AS _ingestion_timestamp,
    uuid() AS _ingestion_run_id,
    sha2(to_json(struct(t.*)), 256) AS _record_hash,
    '1' AS _schema_version
FROM src_transactions t;


In [ ]:
%sql
SELECT * FROM workspace.default.bronze_transactions LIMIT 10;


In [ ]:
%sql
SELECT
    (SELECT COUNT(*) FROM src_transactions) AS source_count,
    (SELECT COUNT(*) FROM workspace.default.bronze_transactions) AS bronze_count,
    CASE
      WHEN (SELECT COUNT(*) FROM src_transactions) =
           (SELECT COUNT(*) FROM workspace.default.bronze_transactions)
      THEN 'MATCH' ELSE 'CHECK'
    END AS status;


# 5. Customers → Bronze

In [ ]:
%sql
CREATE OR REPLACE TABLE workspace.default.bronze_customers
USING DELTA
AS
SELECT
    c.*,
    'customers.json' AS _source_file,
    current_timestamp() AS _ingestion_timestamp,
    uuid() AS _ingestion_run_id,
    sha2(to_json(struct(c.*)), 256) AS _record_hash,
    '1' AS _schema_version
FROM src_customers c;


In [ ]:
%sql
SELECT * FROM workspace.default.bronze_customers LIMIT 10;


In [ ]:
%sql
SELECT
    (SELECT COUNT(*) FROM src_customers) AS source_count,
    (SELECT COUNT(*) FROM workspace.default.bronze_customers) AS bronze_count,
    CASE WHEN (SELECT COUNT(*) FROM src_customers) =
              (SELECT COUNT(*) FROM workspace.default.bronze_customers)
         THEN 'MATCH' ELSE 'CHECK' END AS status;


# 6. Devices → Bronze

In [ ]:
%sql
CREATE OR REPLACE TABLE workspace.default.bronze_devices
USING DELTA
AS
SELECT
    s.*,
    'devices.csv' AS _source_file,
    current_timestamp() AS _ingestion_timestamp,
    uuid() AS _ingestion_run_id,
    sha2(to_json(struct(s.*)), 256) AS _record_hash,
    '1' AS _schema_version
FROM src_devices s;


In [ ]:
%sql
SELECT * FROM workspace.default.bronze_devices LIMIT 10;


In [ ]:
%sql
SELECT
    (SELECT COUNT(*) FROM src_devices) AS source_count,
    (SELECT COUNT(*) FROM workspace.default.bronze_devices) AS bronze_count,
    CASE WHEN (SELECT COUNT(*) FROM src_devices) =
              (SELECT COUNT(*) FROM workspace.default.bronze_devices)
         THEN 'MATCH' ELSE 'CHECK' END AS status;


# 7. Merchants → Bronze

In [ ]:
%sql
CREATE OR REPLACE TABLE workspace.default.bronze_merchants
USING DELTA
AS
SELECT
    s.*,
    'merchants.csv' AS _source_file,
    current_timestamp() AS _ingestion_timestamp,
    uuid() AS _ingestion_run_id,
    sha2(to_json(struct(s.*)), 256) AS _record_hash,
    '1' AS _schema_version
FROM src_merchants s;


In [ ]:
%sql
SELECT * FROM workspace.default.bronze_merchants LIMIT 10;


In [ ]:
%sql
SELECT
    (SELECT COUNT(*) FROM src_merchants) AS source_count,
    (SELECT COUNT(*) FROM workspace.default.bronze_merchants) AS bronze_count,
    CASE WHEN (SELECT COUNT(*) FROM src_merchants) =
              (SELECT COUNT(*) FROM workspace.default.bronze_merchants)
         THEN 'MATCH' ELSE 'CHECK' END AS status;


# 8. Fraud Cases → Bronze

In [ ]:
%sql
CREATE OR REPLACE TABLE workspace.default.bronze_fraud_cases
USING DELTA
AS
SELECT
    s.*,
    'fraud_cases.csv' AS _source_file,
    current_timestamp() AS _ingestion_timestamp,
    uuid() AS _ingestion_run_id,
    sha2(to_json(struct(s.*)), 256) AS _record_hash,
    '1' AS _schema_version
FROM src_fraud_cases s;


In [ ]:
%sql
SELECT * FROM workspace.default.bronze_fraud_cases LIMIT 10;


In [ ]:
%sql
SELECT
    (SELECT COUNT(*) FROM src_fraud_cases) AS source_count,
    (SELECT COUNT(*) FROM workspace.default.bronze_fraud_cases) AS bronze_count,
    CASE WHEN (SELECT COUNT(*) FROM src_fraud_cases) =
              (SELECT COUNT(*) FROM workspace.default.bronze_fraud_cases)
         THEN 'MATCH' ELSE 'CHECK' END AS status;


# 9. Consolidated Source-versus-Bronze Reconciliation

In [ ]:
%sql
SELECT 'transactions' AS dataset,
       (SELECT COUNT(*) FROM src_transactions) AS source_count,
       (SELECT COUNT(*) FROM workspace.default.bronze_transactions) AS bronze_count,
       CASE WHEN (SELECT COUNT(*) FROM src_transactions) =
                 (SELECT COUNT(*) FROM workspace.default.bronze_transactions)
            THEN 'MATCH' ELSE 'CHECK' END AS status
UNION ALL
SELECT 'customers',
       (SELECT COUNT(*) FROM src_customers),
       (SELECT COUNT(*) FROM workspace.default.bronze_customers),
       CASE WHEN (SELECT COUNT(*) FROM src_customers) =
                 (SELECT COUNT(*) FROM workspace.default.bronze_customers)
            THEN 'MATCH' ELSE 'CHECK' END
UNION ALL
SELECT 'devices',
       (SELECT COUNT(*) FROM src_devices),
       (SELECT COUNT(*) FROM workspace.default.bronze_devices),
       CASE WHEN (SELECT COUNT(*) FROM src_devices) =
                 (SELECT COUNT(*) FROM workspace.default.bronze_devices)
            THEN 'MATCH' ELSE 'CHECK' END
UNION ALL
SELECT 'merchants',
       (SELECT COUNT(*) FROM src_merchants),
       (SELECT COUNT(*) FROM workspace.default.bronze_merchants),
       CASE WHEN (SELECT COUNT(*) FROM src_merchants) =
                 (SELECT COUNT(*) FROM workspace.default.bronze_merchants)
            THEN 'MATCH' ELSE 'CHECK' END
UNION ALL
SELECT 'fraud_cases',
       (SELECT COUNT(*) FROM src_fraud_cases),
       (SELECT COUNT(*) FROM workspace.default.bronze_fraud_cases),
       CASE WHEN (SELECT COUNT(*) FROM src_fraud_cases) =
                 (SELECT COUNT(*) FROM workspace.default.bronze_fraud_cases)
            THEN 'MATCH' ELSE 'CHECK' END;


# 10. Verify Bronze Tables

In [ ]:
%sql
SHOW TABLES IN workspace.default;


# 11. Verify Bronze Metadata

In [ ]:
%sql
SELECT
    _source_file,
    _schema_version,
    COUNT(*) AS records,
    MIN(_ingestion_timestamp) AS first_ingestion,
    MAX(_ingestion_timestamp) AS last_ingestion
FROM workspace.default.bronze_transactions
GROUP BY _source_file, _schema_version;


# 12. Safe Repeat-Run Test

Record the count, rerun the Transactions Bronze creation statement, and confirm that `CREATE OR REPLACE TABLE` does not double the data.


In [ ]:
%sql
SELECT COUNT(*) AS count_before_rerun
FROM workspace.default.bronze_transactions;


In [ ]:
%sql
CREATE OR REPLACE TABLE workspace.default.bronze_transactions
USING DELTA
AS
SELECT
    t.*,
    'transactions_databricks_compatible.parquet' AS _source_file,
    current_timestamp() AS _ingestion_timestamp,
    uuid() AS _ingestion_run_id,
    sha2(to_json(struct(t.*)), 256) AS _record_hash,
    '1' AS _schema_version
FROM src_transactions t;


In [ ]:
%sql
SELECT
    (SELECT COUNT(*) FROM src_transactions) AS expected_source_count,
    COUNT(*) AS count_after_rerun,
    CASE
      WHEN COUNT(*) = (SELECT COUNT(*) FROM src_transactions)
      THEN 'SAFE - COUNT UNCHANGED'
      ELSE 'CHECK'
    END AS rerun_status
FROM workspace.default.bronze_transactions;


# 13. Delta Table Detail

In [ ]:
%sql
DESCRIBE DETAIL workspace.default.bronze_transactions;


# 14. Delta History

In [ ]:
%sql
DESCRIBE HISTORY workspace.default.bronze_transactions;


# 15. Week-4 Evidence Checklist

Capture screenshots of:

1. All five batch files visible in the Volume.
2. All five Bronze tables visible in Catalog Explorer / `SHOW TABLES`.
3. Consolidated reconciliation with all five datasets showing `MATCH`.
4. Safe rerun result for transactions.
5. `DESCRIBE HISTORY` for `bronze_transactions`.

Stop after verified Bronze ingestion. Silver, Gold, quarantine, Power BI, Auto Loader and streaming are outside this notebook.
